In [162]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

In [163]:
orders_df = pd.read_csv("../../backend/init_db/orders.csv")
sales_df = pd.read_csv("../../backend/init_db/sales_transaction_lines.csv")

orders_df.head()

,order_id,order_number,supplier_id,product_id,order_date,ordered_qty,expected_delivery_date,actual_delivery_date,received_qty,unit_cost,line_base_cost,discount_pct,tax_rate_pct,status,created_at
0,1,1,2,119,1/1/2024,91,6/1/2024,8/1/2024,86,2.87,261.17,0,8,SENT,1/1/2024 9:00
1,2,1,2,114,1/1/2024,70,6/1/2024,8/1/2024,64,19.10,1337.00,0,8,SENT,1/1/2024 14:00
2,3,1,2,43,1/1/2024,17,6/1/2024,8/1/2024,16,32.91,559.47,0,8,SENT,1/1/2024 13:00
3,4,2,1,71,1/1/2024,26,4/1/2024,5/1/2024,25,18.31,476.06,2,8,RECEIVED,1/1/2024 17:00
4,5,2,1,84,1/1/2024,71,4/1/2024,5/1/2024,68,16.11,1143.81,0,8,RECEIVED,1/1/2024 13:00


In [164]:
orders_df_filter = orders_df[['product_id', 'order_date', 'ordered_qty', 'received_qty']]
orders_df_filter.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1500 entries, 0 to 1499
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   product_id    1500 non-null   int64 
 1   order_date    1500 non-null   object
 2   ordered_qty   1500 non-null   int64 
 3   received_qty  1500 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 47.0+ KB


In [165]:
sales_df.head()

,sale_item_id,sale_id,product_id,line_number,base_unit_price,sale_unit_price,quantity,line_base_amount,line_sale_amount,line_discount_amount,tax_rate_pct,tax_amount,promotion_code,created_at
0,1,1,76,1,52.38,52.38,2,104.76,104.76,0.00,8,8.38,NaN,15/9/2024 13:15
1,2,1,143,2,17.09,14.53,1,17.09,14.53,2.56,8,1.16,MEMBER15,15/9/2024 13:15
2,3,1,8,3,18.08,18.08,2,36.16,36.16,0.00,8,2.89,NaN,15/9/2024 13:15
3,4,1,130,4,41.12,41.12,2,82.24,82.24,0.00,8,6.58,NaN,15/9/2024 13:15
4,5,2,31,1,28.96,27.51,1,28.96,27.51,1.45,8,2.20,BULK5,30/4/2024 10:49


In [166]:
sales_df_filter = sales_df[['product_id', 'quantity', 'created_at']]
sales_df_filter.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19862 entries, 0 to 19861
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   product_id  19862 non-null  int64 
 1   quantity    19862 non-null  int64 
 2   created_at  19862 non-null  object
dtypes: int64(2), object(1)
memory usage: 465.6+ KB


In [167]:
orders_df_filter

,product_id,order_date,ordered_qty,received_qty
0,119,1/1/2024,91,86
1,114,1/1/2024,70,64
2,43,1/1/2024,17,16
3,71,1/1/2024,26,25
4,84,1/1/2024,71,68
...,...,...,...,...
1495,98,27/5/2025,46,46
1496,114,27/5/2025,36,33
1497,137,27/5/2025,76,74
1498,116,27/5/2025,68,63


In [168]:
orders_df_filter['mth'] = orders_df_filter['order_date'].str.split('/').str[1]
orders_df_filter['year'] = orders_df_filter['order_date'].str.split('/').str[2]

orders_df_filter['mth'] = orders_df_filter['mth'].astype('int')
orders_df_filter['year'] = orders_df_filter['year'].astype('int')

orders_df_filter = orders_df_filter[['mth', 'year', 'product_id', 'ordered_qty']]

orders_df_filter

C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\1048376824.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_df_filter['mth'] = orders_df_filter['order_date'].str.split('/').str[1]
C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\1048376824.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  orders_df_filter['year'] = orders_df_filter['order_date'].str.split('/').str[2]
C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\1048376824.py:4: SettingWithCopyWarning: 
A value is trying to

,mth,year,product_id,ordered_qty
0,1,2024,119,91
1,1,2024,114,70
2,1,2024,43,17
3,1,2024,71,26
4,1,2024,84,71
...,...,...,...,...
1495,5,2025,98,46
1496,5,2025,114,36
1497,5,2025,137,76
1498,5,2025,116,68


In [169]:
orders = orders_df_filter.groupby(['product_id', 'mth', 'year'])[['ordered_qty']].sum().reset_index()

print(orders)


      product_id  mth  year  ordered_qty
0              1    2  2025           40
1              1    3  2024          125
2              1    3  2025           50
3              1    4  2025           47
4              1    5  2024          113
...          ...  ...   ...          ...
1074         148    6  2024          309
1075         148    7  2024           80
1076         148    9  2024          658
1077         148   11  2024          384
1078         148   12  2024          259

[1079 rows x 4 columns]


In [170]:
sales_df_filter

,product_id,quantity,created_at
0,76,2,15/9/2024 13:15
1,143,1,15/9/2024 13:15
2,8,2,15/9/2024 13:15
3,130,2,15/9/2024 13:15
4,31,1,30/4/2024 10:49
...,...,...,...
19857,126,3,15/7/2024 13:51
19858,107,4,21/1/2024 15:27
19859,140,3,21/1/2024 15:27
19860,82,3,21/1/2024 15:27


In [171]:
sales_df_filter['datetime'] = sales_df_filter['created_at'].str.split(' ').str[0]
sales_df_filter['mth'] = sales_df_filter['datetime'].str.split('/').str[1]
sales_df_filter['year'] = sales_df_filter['datetime'].str.split('/').str[2]

sales_df_filter['mth'] = sales_df_filter['mth'].astype('int')
sales_df_filter['year'] = sales_df_filter['year'].astype('int')

sales_df_filter = sales_df_filter[['mth', 'year', 'product_id', 'quantity']]
sales_df_filter

C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\4166366081.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_filter['datetime'] = sales_df_filter['created_at'].str.split(' ').str[0]
C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\4166366081.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sales_df_filter['mth'] = sales_df_filter['datetime'].str.split('/').str[1]
C:\Users\Caden Tang\AppData\Local\Temp\ipykernel_23540\4166366081.py:3: SettingWithCopyWarning: 
A value is trying to b

,mth,year,product_id,quantity
0,9,2024,76,2
1,9,2024,143,1
2,9,2024,8,2
3,9,2024,130,2
4,4,2024,31,1
...,...,...,...,...
19857,7,2024,126,3
19858,1,2024,107,4
19859,1,2024,140,3
19860,1,2024,82,3


In [172]:
sales = sales_df_filter.groupby(['product_id', 'mth', 'year'])[['quantity']].sum().reset_index()

sales


,product_id,mth,year,quantity
0,1,1,2024,15
1,1,1,2025,7
2,1,2,2024,20
3,1,2,2025,6
4,1,3,2024,16
...,...,...,...,...
2537,150,8,2024,1
2538,150,9,2024,4
2539,150,10,2024,5
2540,150,11,2024,6


In [173]:
merged_df = pd.merge(orders, sales, on=['mth', 'year', 'product_id'], how='inner')

merged_df = merged_df.rename(columns={
    'quantity': 'sales_qty',
    'qty': 'order_qty'
})

merged_df

,product_id,mth,year,ordered_qty,sales_qty
0,1,2,2025,40,6
1,1,3,2024,125,16
2,1,3,2025,50,13
3,1,4,2025,47,16
4,1,5,2024,113,6
...,...,...,...,...,...
1069,148,6,2024,309,34
1070,148,7,2024,80,35
1071,148,9,2024,658,28
1072,148,11,2024,384,22


In [174]:
merged_df['product_id'] = merged_df['product_id'].astype('int')
merged_df['mth'] = merged_df['mth'].astype('int')
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1074 entries, 0 to 1073
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   product_id   1074 non-null   int32
 1   mth          1074 non-null   int32
 2   year         1074 non-null   int32
 3   ordered_qty  1074 non-null   int64
 4   sales_qty    1074 non-null   int64
dtypes: int32(3), int64(2)
memory usage: 29.5 KB


In [175]:
X_train, X_test, y_train, y_test = train_test_split(merged_df.drop('ordered_qty', axis=1),
                                                    merged_df['ordered_qty'],
                                                    test_size=0.25,
                                                    random_state=42)

(X_train.shape, X_test.shape)

((805, 4), (269, 4))

In [176]:
lr = LinearRegression()
lr.fit(X_train, y_train)

LinearRegression()

In [177]:
y_pred = lr.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"R-squared (R2 Score): {r2:.4f}")

Mean Squared Error (MSE): 10153.3119
R-squared (R2 Score): 0.0389


In [178]:
merged_df['ordered_qty'].describe()

count    1074.000000
mean      112.770019
std        93.366648
min        11.000000
25%        54.000000
50%        87.000000
75%       141.750000
max       869.000000
Name: ordered_qty, dtype: float64

In [179]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Quick test with a tree model
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(f"RF Test R2: {r2_score(y_test, rf_preds):.4f}")
print(f"RF Test RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.4f}")

RF Test R2: 0.0865
RF Test RMSE: 98.2387


In [180]:
import pandas as pd

# 1. Force the IDs and months to be interpreted as categories/text
merged_df['product_id'] = merged_df['product_id'].astype(str)
merged_df['mth'] = merged_df['mth'].astype(str)

# 2. Convert these categories into 0s and 1s (One-Hot Encoding)
X = pd.get_dummies(merged_df[['product_id', 'mth','sales_qty']], drop_first=True)
y = merged_df['ordered_qty']

# Convert boolean True/False outputs from get_dummies into 1 and 0 numbers
X = X.astype(float)

In [181]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

# Quick test with a tree model
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)
rf_preds = rf_model.predict(X_test)

print(f"RF Test R2: {r2_score(y_test, rf_preds):.4f}")
print(f"RF Test RMSE: {np.sqrt(mean_squared_error(y_test, rf_preds)):.4f}")

RF Test R2: 0.1421
RF Test RMSE: 95.2034


In [182]:
import joblib

model_filename = 'rfr_order_model.joblib'

# Save the model to your storage drive
joblib.dump(rf_model, model_filename)

print(f"Model successfully saved to {model_filename}")

Model successfully saved to rfr_order_model.joblib


In [183]:
merged_df[(merged_df['year']==2025) & (merged_df['mth']=='5')]

,product_id,mth,year,ordered_qty,sales_qty
11,4,5,2025,91,17
24,6,5,2025,26,15
76,14,5,2025,136,1
85,15,5,2025,72,22
100,17,5,2025,74,9
...,...,...,...,...,...
1002,138,5,2025,27,9
1035,143,5,2025,95,2
1046,144,5,2025,167,15
1056,145,5,2025,54,35
